# HJB PDE Solver (Grid-Based, d=2)

Solve the full Hamilton–Jacobi–Bellman PDE (Eq. 1 in Barzykin, Bergault, Guéant 2023) for the market maker's value function $\theta(t,y)$ on a 2-currency (USD–EUR) inventory grid via explicit Euler time-stepping.

**Method:** Backward-in-time explicit Euler. At each step the spatial RHS is assembled from:
- Running penalty: $-\frac{\gamma}{2} y^\top \Sigma y$
- Diffusion: $\frac{1}{2} \mathrm{Tr}(D(y)\Sigma D(y)\, D^2\theta)$
- Drift: $\sum_i \mu_i y_i (1 + \partial\theta/\partial y_i)$
- Quoting Hamiltonian: sum over all (pair, tier, size) contributions
- Hedging Hamiltonian: $H^{i,j}(p)$ with execution cost $L(\xi) = \psi|\xi| + \eta\xi^2$

**Stability note:** The paper's $\eta$ values (e.g. $\eta_{\text{EURUSD}} = 10^{-9}$ decimal) make the hedging Hamiltonian extremely stiff for explicit Euler. We scale $\eta$ up by a factor of 1000 for the PDE solve. The ODE reference solution uses the original (unscaled) parameters.

## 1. Imports and Parameter Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.model import (
    build_paper_example_params, restrict_currencies,
    ModelParams, PairParams, BP, canon_pair,
)
from src.hamiltonian import optimal_delta_logistic, H_logistic
from src.riccati import build_Sigma
from src.policy import (
    optimal_client_markup, optimal_hedge_rate,
    run_multicurrency_mm,
)
from src.pde import (
    solve_hjb_explicit, compute_gradient, build_pde_spec,
    H_execution_cost,
)

# --- Build parameters ---
# Original (unscaled) for ODE reference
mp = restrict_currencies(build_paper_example_params(), ["USD", "EUR"])

# Scaled eta for PDE stability
ETA_SCALE = 1000
mp_pde = restrict_currencies(build_paper_example_params(), ["USD", "EUR"])
new_pairs = {}
for key, pp in mp_pde.pairs.items():
    new_pairs[key] = PairParams(
        pair=pp.pair,
        sizes_musd=pp.sizes_musd,
        lambdas_per_day=pp.lambdas_per_day,
        tiers=pp.tiers,
        psi=pp.psi,
        eta=pp.eta * ETA_SCALE,
    )
mp_pde.pairs = new_pairs

print(f"Currencies: {mp_pde.currencies}")
print(f"d = {len(mp_pde.currencies)}")
print(f"T = {mp_pde.T_days} days")
print(f"gamma = {mp_pde.gamma}")
eurusd_key = canon_pair('EUR', 'USD')
print(f"eta (original): {mp.pairs[eurusd_key].eta:.2e}")
print(f"eta (scaled):   {mp_pde.pairs[eurusd_key].eta:.2e}")
print(f"psi:            {mp_pde.pairs[eurusd_key].psi:.2e}")

## 2. Grid Construction

In [ ]:
Y_MAX = 150
DY = 1.0
d = len(mp_pde.currencies)

y_grids = [np.arange(-Y_MAX, Y_MAX + DY, DY) for _ in range(d)]

print(f"Number of currencies (d): {d}")
print(f"Grid range: [{-Y_MAX}, {Y_MAX}] M$")
print(f"Grid spacing: {DY} M$")
print(f"Points per axis: {len(y_grids[0])}")
print(f"Total grid points: {len(y_grids[0])**d}")
print(f"Index of y=0: {np.searchsorted(y_grids[0], 0.0)}")

## 3. Run PDE Solver

In [ ]:
N_STEPS = 1000
T = mp_pde.T_days

snapshot_times = [T * 0.75, T * 0.5, T * 0.25, T * 0.1]
print(f"Time step dt = {T / N_STEPS:.6e} days")
print(f"Snapshot times: {[f'{t:.4f}' for t in snapshot_times]}")
print("Running PDE solver...")

result = solve_hjb_explicit(y_grids, mp_pde, n_steps=N_STEPS, snapshot_times=snapshot_times)

theta_0 = result['theta_0']
snapshots = result['snapshots']
spec = result['spec']
dt = result['dt']

print(f"\ntheta_0 shape: {theta_0.shape}")
print(f"theta_0 range: [{theta_0.min():.4f}, {theta_0.max():.4f}]")
idx_origin = len(y_grids[0]) // 2  # index of y=0
print(f"theta_0 at origin: {theta_0[idx_origin, idx_origin]:.6f}")
print(f"All finite: {np.all(np.isfinite(theta_0))}")
print(f"Snapshots collected: {sorted(snapshots.keys())}")

## 4. Run ODE Reference Solution

In [ ]:
# ODE uses ORIGINAL (unscaled) parameters
ode_res = run_multicurrency_mm(mp)
A0 = ode_res.A0
B0 = ode_res.B0

print("ODE solution (original parameters):")
print(f"A0 =\n{A0}")
print(f"B0 = {B0}")
print(f"\nNote: B0 = 0 because parameters are direction-symmetric.")

## 5. Value Function Comparison

Slice $\theta_{\text{PDE}}(0, y)$ along the EUR axis with $y_{\text{USD}} = 0$, and compare against the ODE ansatz $\hat{\theta} = -y^\top A_0 y - y^\top B_0$, shifted to match the PDE at the origin (absorbing the constant $C_0$).

Since we use scaled $\eta$ for the PDE, the hedging contribution differs quantitatively. The comparison shows agreement in the quadratic structure from quoting and risk terms.

In [ ]:
# Slice along EUR axis (index 1), y_USD=0 (index 0 at idx_origin)
y_eur = y_grids[1]
theta_pde_slice = theta_0[idx_origin, :]  # y_USD=0, varying y_EUR

# ODE ansatz: theta_hat = -y^T A0 y - y^T B0
# With y = (0, y_EUR): theta_hat = -A0[1,1]*y_EUR^2 - B0[1]*y_EUR
theta_ode_raw = -A0[1, 1] * y_eur**2 - B0[1] * y_eur

# Shift to match PDE at origin (absorb C0)
C0_shift = theta_pde_slice[idx_origin] - theta_ode_raw[idx_origin]
theta_ode_shifted = theta_ode_raw + C0_shift

print(f"C0 shift (PDE origin - ODE origin): {C0_shift:.6f}")

# Plot region of interest
roi = np.abs(y_eur) <= 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(y_eur[roi], theta_pde_slice[roi], 'b-', linewidth=2, label='PDE (scaled $\\eta$)')
ax1.plot(y_eur[roi], theta_ode_shifted[roi], 'r--', linewidth=2, label='ODE ansatz (original $\\eta$, shifted)')
ax1.set_xlabel('$y_{\\mathrm{EUR}}$ (M\$)', fontsize=12)
ax1.set_ylabel('$\\theta(0, y)$', fontsize=12)
ax1.set_title('Value function: PDE vs ODE', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

diff = theta_pde_slice[roi] - theta_ode_shifted[roi]
ax2.plot(y_eur[roi], diff, 'k-', linewidth=1.5)
ax2.set_xlabel('$y_{\\mathrm{EUR}}$ (M\$)', fontsize=12)
ax2.set_ylabel('$\\theta_{\\mathrm{PDE}} - \\hat{\\theta}_{\\mathrm{ODE}}$', fontsize=12)
ax2.set_title('Difference (PDE $-$ ODE)', fontsize=13)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"Max absolute difference in [-100, 100]: {np.max(np.abs(diff)):.6f}")

## 6. Policy Comparison — Optimal Quotes

Extract optimal bid/ask quotes from the PDE solution and compare against the ODE approximation.

For the PDE, the optimal markup for a trade where the client pays EUR, sells USD (tier 0, $z = 1$ M\$) is:
$$p(y) = \frac{\theta(y) - \theta(y + z(e_{\text{EUR}} - e_{\text{USD}}))}{z}, \quad \delta^* = \arg\max_\delta f(\delta)(\delta - p)$$

Note: PDE quotes use `mp_pde` parameters (the PDE was solved with scaled $\eta$). ODE quotes use original `mp`.

In [ ]:
# PDE quote extraction for EURUSD tier 0, z=1 M$
# Direction: pay EUR (index 1), sell USD (index 0)
# Shift: +z on EUR axis (index 1), -z on USD axis (index 0)
z = 1.0  # M$
shift_eur = int(round(z / DY))   # +1 on EUR axis
shift_usd = int(round(-z / DY))  # -1 on USD axis

# Get tier parameters from mp_pde (what the PDE solved with)
pp_pde = mp_pde.pairs[eurusd_key]
tier0_pde = pp_pde.tiers[0]

# Slice: y_USD=0 (index idx_origin on axis 0)
# For each y_EUR, look up theta(y_USD=0, y_EUR) and theta(y_USD=0+shift_usd, y_EUR+shift_eur)
# shift_usd = -1, shift_eur = +1
n_eur = len(y_grids[1])

# Valid range: need idx_origin + shift_usd >= 0 and y_eur_idx + shift_eur < n_eur
# shift_usd = -1: need idx_origin >= 1 (yes, 150 >= 1)
# shift_eur = +1: need y_eur_idx <= n_eur - 2
eur_valid = slice(0, n_eur - shift_eur)  # indices where shifted lookup is in bounds

theta_here = theta_0[idx_origin, eur_valid]
theta_shifted = theta_0[idx_origin + shift_usd, shift_eur:n_eur]

p_pde = (theta_here - theta_shifted) / z
delta_pde = optimal_delta_logistic(p_pde, tier0_pde.alpha, tier0_pde.beta)
y_eur_valid = y_grids[1][eur_valid]

# ODE reference quotes (original parameters)
pp_orig = mp.pairs[eurusd_key]
tier0_orig = pp_orig.tiers[0]
delta_ode = np.array([
    optimal_client_markup(
        mp, tier_idx=0, ccy_pay='EUR', ccy_sell='USD',
        z_musd=z, y=np.array([0.0, ye]), A=A0, B=B0
    )
    for ye in y_eur_valid
])

# Convert to basis points
delta_pde_bps = delta_pde / BP
delta_ode_bps = delta_ode / BP

# Plot in region of interest
roi_q = np.abs(y_eur_valid) <= 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(y_eur_valid[roi_q], delta_pde_bps[roi_q], 'b-', linewidth=2, label='PDE (scaled $\\eta$)')
ax1.plot(y_eur_valid[roi_q], delta_ode_bps[roi_q], 'r--', linewidth=2, label='ODE (original $\\eta$)')
ax1.set_xlabel('$y_{\\mathrm{EUR}}$ (M\$)', fontsize=12)
ax1.set_ylabel('Optimal markup $\\delta^*$ (bps)', fontsize=12)
ax1.set_title('EURUSD Tier 0, z=1 M\$: Optimal quote', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

diff_q = delta_pde_bps[roi_q] - delta_ode_bps[roi_q]
ax2.plot(y_eur_valid[roi_q], diff_q, 'k-', linewidth=1.5)
ax2.set_xlabel('$y_{\\mathrm{EUR}}$ (M\$)', fontsize=12)
ax2.set_ylabel('$\\delta^*_{\\mathrm{PDE}} - \\delta^*_{\\mathrm{ODE}}$ (bps)', fontsize=12)
ax2.set_title('Quote difference (PDE $-$ ODE)', fontsize=13)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"Max quote difference in [-100, 100]: {np.max(np.abs(diff_q)):.4f} bps")

## 7. Policy Comparison — Optimal Hedge Rates

Extract optimal hedging rates from the PDE gradient and compare against the ODE formula.

For the PDE, the hedging momentum for EURUSD is:
$$p = \frac{\partial\theta}{\partial y_{\text{EUR}}} - \frac{\partial\theta}{\partial y_{\text{USD}}} + k_{\text{EUR}} y_{\text{EUR}} \Big(1 + \frac{\partial\theta}{\partial y_{\text{EUR}}}\Big) - k_{\text{USD}} y_{\text{USD}} \Big(1 + \frac{\partial\theta}{\partial y_{\text{USD}}}\Big)$$
$$\xi^* = H'(p) = \frac{\mathrm{sign}(p) \max(|p| - \psi, 0)}{2\eta}$$

The PDE uses scaled $\eta$, so hedge rates will differ quantitatively from the ODE (which uses original $\eta$).

In [ ]:
# Compute gradient of theta_0
dy_list = [DY, DY]
grad_theta = compute_gradient(theta_0, dy_list)
# grad_theta[0] = d(theta)/d(y_USD), grad_theta[1] = d(theta)/d(y_EUR)

# Market impact coefficients
k_usd = mp_pde.k.get('USD', 0.0)
k_eur = mp_pde.k.get('EUR', 0.0)

# Hedging parameters from mp_pde (scaled eta)
psi_pde = pp_pde.psi
eta_pde = pp_pde.eta

# Slice at y_USD = 0
dtheta_dusd = grad_theta[0][idx_origin, :]  # d(theta)/d(y_USD) at y_USD=0
dtheta_deur = grad_theta[1][idx_origin, :]  # d(theta)/d(y_EUR) at y_USD=0

# Hedging momentum: buy EUR (i=1), sell USD (j=0)
# p = grad_theta[1] - grad_theta[0] + k_EUR * y_EUR * (1 + grad_theta[1]) - k_USD * y_USD * (1 + grad_theta[0])
# At y_USD = 0:
p_hedge_pde = (dtheta_deur - dtheta_dusd
               + k_eur * y_eur * (1.0 + dtheta_deur)
               - k_usd * 0.0 * (1.0 + dtheta_dusd))

# H'(p) = sign(p) * max(|p| - psi, 0) / (2 * eta)
excess_pde = np.maximum(np.abs(p_hedge_pde) - psi_pde, 0.0)
xi_pde = np.sign(p_hedge_pde) * excess_pde / (2.0 * eta_pde)

# ODE reference hedge rates (original parameters)
xi_ode = np.array([
    optimal_hedge_rate(
        mp, ccy_buy='EUR', ccy_sell='USD',
        y=np.array([0.0, ye]), A=A0, B=B0
    )
    for ye in y_eur
])

# Plot
roi_h = np.abs(y_eur) <= 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(y_eur[roi_h], xi_pde[roi_h], 'b-', linewidth=2, label='PDE (scaled $\\eta$)')
ax1.plot(y_eur[roi_h], xi_ode[roi_h], 'r--', linewidth=2, label='ODE (original $\\eta$)')
ax1.set_xlabel('$y_{\\mathrm{EUR}}$ (M\$)', fontsize=12)
ax1.set_ylabel('$\\xi^*$ (M\$/day)', fontsize=12)
ax1.set_title('EURUSD Hedge rate vs EUR inventory', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(y_eur[roi_h], xi_pde[roi_h], 'b-', linewidth=2, label='PDE (scaled $\\eta$)')
ax2.set_xlabel('$y_{\\mathrm{EUR}}$ (M\$)', fontsize=12)
ax2.set_ylabel('$\\xi^*_{\\mathrm{PDE}}$ (M\$/day)', fontsize=12)
ax2.set_title('PDE hedge rate (detail)', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"PDE hedge rate range: [{xi_pde[roi_h].min():.2f}, {xi_pde[roi_h].max():.2f}] M$/day")
print(f"ODE hedge rate range: [{xi_ode[roi_h].min():.2f}, {xi_ode[roi_h].max():.2f}] M$/day")
print(f"Note: ODE rates are ~{ETA_SCALE}x larger due to {ETA_SCALE}x smaller eta.")

## 8. Stationarity Check

Plot $\theta(t, y)$ slices at several snapshot times and at $t=0$ along the EUR axis ($y_{\text{USD}}=0$). As $t \to 0$ (far from terminal time), the solution should converge to a stationary profile (curves collapsing).

In [ ]:
# Collect all snapshots + t=0 for plotting
times_sorted = sorted(snapshots.keys(), reverse=True)
all_slices = {}

for t_snap in times_sorted:
    theta_snap = snapshots[t_snap]
    all_slices[t_snap] = theta_snap[idx_origin, :]

all_slices[0.0] = theta_0[idx_origin, :]

# Shift all curves so they match at origin (absorb different C(t) values)
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(all_slices)))

roi_s = np.abs(y_eur) <= 80

for idx, (t_snap, sl) in enumerate(sorted(all_slices.items(), reverse=True)):
    # Shift to match at origin for comparison of curvature
    sl_shifted = sl - sl[idx_origin]
    ax.plot(y_eur[roi_s], sl_shifted[roi_s], color=colors[idx],
            linewidth=2, label=f't = {t_snap:.4f} days')

ax.set_xlabel('$y_{\\mathrm{EUR}}$ (M\$)', fontsize=12)
ax.set_ylabel('$\\theta(t, y) - \\theta(t, 0)$', fontsize=12)
ax.set_title('Stationarity check: value function at different times', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

# Print max changes between consecutive snapshots
print("Max |theta(t1) - theta(t2)| between consecutive snapshots (shifted):")
all_times = sorted(all_slices.keys(), reverse=True)
for i in range(len(all_times) - 1):
    t1, t2 = all_times[i], all_times[i + 1]
    s1 = all_slices[t1] - all_slices[t1][idx_origin]
    s2 = all_slices[t2] - all_slices[t2][idx_origin]
    max_diff = np.max(np.abs(s1 - s2))
    print(f"  t={t1:.4f} -> t={t2:.4f}: {max_diff:.6f}")

## 9. Stability Diagnostics

Summary of solver diagnostics and the role of $\eta$ scaling.

In [ ]:
print("=== Stability Diagnostics ===")
print(f"theta_0 all finite: {np.all(np.isfinite(theta_0))}")
print(f"theta_0 range:      [{theta_0.min():.4f}, {theta_0.max():.4f}]")
print(f"theta_0 at origin:  {theta_0[idx_origin, idx_origin]:.6f}")
print(f"theta_0 max |value|: {np.max(np.abs(theta_0)):.4f}")
print()
print(f"Grid: {len(y_grids[0])}x{len(y_grids[1])}, dy={DY}, Y_MAX={Y_MAX}")
print(f"Time: N_STEPS={N_STEPS}, dt={dt:.6e} days, T={T} days")
print(f"ETA_SCALE = {ETA_SCALE}")
print()
print("Note: Explicit Euler is CFL-unstable with the paper's eta values.")
print(f"Scaling eta by {ETA_SCALE}x reduces the hedging Hamiltonian's")
print(f"contribution by the same factor, keeping the scheme stable.")
print(f"For production use, an implicit or semi-implicit scheme would")
print(f"handle the original parameters without rescaling.")